# ARC-v0.13 — FEVER Boundary External Replication

This notebook is a **cross-dataset external-validity replication** of the ARC-v0.9
HotpotQA boundary/stability analysis.

The design is intentionally frozen before examining FEVER boundary outcomes.

Primary questions:

1. Does the PQ32↔SQ8 approximation-feedback boundary remain heterogeneous on FEVER?
2. Can the same first-order baseline feature family predict high H3 amplification out of sample?
3. Are stable/null and reversal cases retained?
4. How close are FEVER held-out discrimination and regime structure to HotpotQA ARC-v0.9?

This notebook does **not** claim that FEVER reproduces HotpotQA automatically.
A null or weaker replication is retained as a valid result.

TEST remains untouched.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
%pip install -q faiss-cpu==1.12.0 psutil pyarrow scikit-learn pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 52.8 MB/s eta 0:00:00


In [5]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import gc
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import faiss
import psutil

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    roc_auc_score,
    average_precision_score,
    accuracy_score,
)

print("FAISS:", faiss.__version__)
print("NumPy:", np.__version__)
print("RAM GiB:", psutil.virtual_memory().total / 1024**3)

FAISS: 1.12.0
NumPy: 2.0.2
RAM GiB: 12.671417236328125


## 1. Frozen paths and provenance

In [6]:
SEED = 20260816
DIM = 384
N_DOCS = 5_416_568
NPROBE = 64
TOP_RETRIEVE = 100
TOP_K = 10
MAX_ROUNDS = 4

ROOT = Path(
    "/content/drive/MyDrive/"
    "hc-rars-fever-5m-untouched-confirmation-v1"
)

ARC_ROOT = Path(
    "/content/drive/MyDrive/"
    "rag-pq-checkpoints/arc-v0"
)

INDEX_ROOT = Path(
    "/content/drive/MyDrive/"
    "rag-pq-checkpoints/arc-index-cache"
)

CORPUS_MEMMAP = (
    ROOT / "stage1/corpus_embeddings.float16.memmap"
)

QUERY_EMB = (
    ROOT / "stage1/query_embeddings_v2.float32.npy"
)

QUERY_IDS = (
    ROOT / "stage1/query_ids.utf8.txt"
)

SPLIT_MANIFEST = (
    ROOT / "stage1/official_split_manifest.json"
)

DEV_QRELS = (
    ROOT / "stage2/dev_qrels_rows.csv"
)

PQ32_PATH = (
    INDEX_ROOT
    / "fever5m-bge-small-ivfpq-nlist4096-m32-nbits8-seed20260816.faiss"
)

SQ8_PATH = (
    INDEX_ROOT
    / "fever5m-bge-small-ivfsq8-nlist4096-seed20260816.faiss"
)

# HotpotQA v0.9 is used only as an external reference after the FEVER protocol is sealed.
HOTPOT_V09_ROOT = (
    ARC_ROOT / "boundary-stability-map-v09"
)

required = {
    "CORPUS_MEMMAP": CORPUS_MEMMAP,
    "QUERY_EMB": QUERY_EMB,
    "QUERY_IDS": QUERY_IDS,
    "SPLIT_MANIFEST": SPLIT_MANIFEST,
    "DEV_QRELS": DEV_QRELS,
    "PQ32_INDEX": PQ32_PATH,
    "SQ8_INDEX": SQ8_PATH,
}

for name, path in required.items():
    print(f"{name:20s}", "OK" if path.is_file() else "MISSING", path)
    if not path.is_file():
        raise FileNotFoundError(path)

print("\nFEVER PREFLIGHT — PASS")

CORPUS_MEMMAP        OK /content/drive/MyDrive/hc-rars-fever-5m-untouched-confirmation-v1/stage1/corpus_embeddings.float16.memmap
QUERY_EMB            OK /content/drive/MyDrive/hc-rars-fever-5m-untouched-confirmation-v1/stage1/query_embeddings_v2.float32.npy
QUERY_IDS            OK /content/drive/MyDrive/hc-rars-fever-5m-untouched-confirmation-v1/stage1/query_ids.utf8.txt
SPLIT_MANIFEST       OK /content/drive/MyDrive/hc-rars-fever-5m-untouched-confirmation-v1/stage1/official_split_manifest.json
DEV_QRELS            OK /content/drive/MyDrive/hc-rars-fever-5m-untouched-confirmation-v1/stage2/dev_qrels_rows.csv
PQ32_INDEX           OK /content/drive/MyDrive/rag-pq-checkpoints/arc-index-cache/fever5m-bge-small-ivfpq-nlist4096-m32-nbits8-seed20260816.faiss
SQ8_INDEX            OK /content/drive/MyDrive/rag-pq-checkpoints/arc-index-cache/fever5m-bge-small-ivfsq8-nlist4096-seed20260816.faiss

FEVER PREFLIGHT — PASS


## 2. Seal the FEVER boundary-replication protocol

In [7]:
def sha256_file(path, chunk_size=64 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

OUT_ROOT = (
    ARC_ROOT / "fever-boundary-external-replication-v013"
)
OUT_ROOT.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
OUT = OUT_ROOT / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

MEAN_ALPHAS = [0.1, 0.3, 0.5, 0.7]
MEAN_KS = [5, 20, 50]

SOFTMAX_ALPHAS = [0.1, 0.3, 0.5, 0.7]
SOFTMAX_KS = [5, 20]
SOFTMAX_TEMPS = [0.05, 0.1, 0.2, 0.5]

BOUNDARY_CONFIGS = []

for alpha in MEAN_ALPHAS:
    for k in MEAN_KS:
        BOUNDARY_CONFIGS.append({
            "method": "mean",
            "alpha": alpha,
            "k": k,
            "temperature": None,
        })

for alpha in SOFTMAX_ALPHAS:
    for k in SOFTMAX_KS:
        for temperature in SOFTMAX_TEMPS:
            BOUNDARY_CONFIGS.append({
                "method": "softmax",
                "alpha": alpha,
                "k": k,
                "temperature": temperature,
            })

assert len(BOUNDARY_CONFIGS) == 44

protocol = {
    "status": "FEVER_BOUNDARY_EXTERNAL_REPLICATION_SEALED_BEFORE_SWEEP",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": "FEVER",
    "split": "DEV",
    "expected_dev_queries": 6666,
    "encoder": "BAAI/bge-small-en-v1.5",
    "corpus_rows": N_DOCS,
    "dimension": DIM,
    "retriever_pair": ["PQ32", "SQ8"],
    "nprobe": NPROBE,
    "top_retrieve": TOP_RETRIEVE,
    "feedback_rounds": MAX_ROUNDS,
    "query_split": "deterministic SHA256 parity 50/50 fit-validation",
    "boundary_grid_config_count": len(BOUNDARY_CONFIGS),
    "boundary_configs": BOUNDARY_CONFIGS,
    "prediction_target": "H3_slope",
    "high_amplification_definition":
        "FIT-family 75th percentile of H3_slope; threshold frozen before validation",
    "regime_threshold_abs_slope": 0.002,
    "retain_null_and_reversal_cases": True,
    "test_access_allowed": False,
    "hotpotqa_v09_used_for_parameter_tuning": False,
}

raw = json.dumps(
    protocol,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

protocol_sha = hashlib.sha256(raw).hexdigest()
protocol["protocol_sha256"] = protocol_sha

PROTOCOL_PATH = OUT / "v013_fever_boundary_protocol.json"
PROTOCOL_PATH.write_text(
    json.dumps(protocol, indent=2),
    encoding="utf-8",
)

print("Output:", OUT)
print("Protocol SHA-256:", protocol_sha)
print("ARC-v0.13 PROTOCOL — SEALED")

Output: /content/drive/MyDrive/rag-pq-checkpoints/arc-v0/fever-boundary-external-replication-v013/20260817-151852
Protocol SHA-256: cc56579a1796f2a2372e9fdc17ebf79a535c083dc1939ec8d69fb8056acf260f
ARC-v0.13 PROTOCOL — SEALED


## 3. Load FEVER DEV queries, qrels, and corpus embeddings

In [8]:
queries = np.load(
    QUERY_EMB,
    mmap_mode="r",
)

with open(QUERY_IDS, "r", encoding="utf-8") as f:
    query_ids = [x.strip() for x in f if x.strip()]

with open(SPLIT_MANIFEST, "r", encoding="utf-8") as f:
    split = json.load(f)

dev_ids = [str(x) for x in split["dev_query_ids"]]

assert len(dev_ids) == 6666

for key in [
    "test_qrels_relevance_values_accessed",
    "test_retrieval_performed",
    "test_outcomes_observed",
]:
    if key in split:
        assert split[key] is False, (key, split[key])

query_row = {
    str(qid): i
    for i, qid in enumerate(query_ids)
}

missing = [q for q in dev_ids if q not in query_row]
assert not missing, missing[:10]

dev_rows = np.asarray(
    [query_row[q] for q in dev_ids],
    dtype=np.int64,
)

Q_DEV = np.asarray(
    queries[dev_rows],
    dtype=np.float32,
)

Q_DEV /= np.maximum(
    np.linalg.norm(Q_DEV, axis=1, keepdims=True),
    1e-12,
)

qrels_df = pd.read_csv(DEV_QRELS)

print("qrels columns:", qrels_df.columns.tolist())

# Flexible schema normalization for the already-generated FEVER row-qrels artifact.
qid_col = next(
    c for c in ["query_id", "query-id", "qid"]
    if c in qrels_df.columns
)

row_col = next(
    c for c in ["corpus_row", "corpus-row", "row_id", "doc_row"]
    if c in qrels_df.columns
)

score_candidates = [
    c for c in ["score", "relevance", "rel"]
    if c in qrels_df.columns
]

qrels_df[qid_col] = qrels_df[qid_col].astype(str)
qrels_df[row_col] = qrels_df[row_col].astype(np.int64)

if score_candidates:
    score_col = score_candidates[0]
    positive = qrels_df[qrels_df[score_col] > 0]
else:
    positive = qrels_df

dev_qrels = {
    str(qid): set(
        g[row_col].astype(np.int64).tolist()
    )
    for qid, g in positive.groupby(qid_col)
}

assert all(q in dev_qrels for q in dev_ids)

CORPUS = np.memmap(
    CORPUS_MEMMAP,
    dtype=np.float16,
    mode="r",
    shape=(N_DOCS, DIM),
)

print("Q_DEV:", Q_DEV.shape)
print("Corpus:", CORPUS.shape)
print("DEV qrels:", len(dev_qrels))
print("FEVER DEV ALIGNMENT — PASS")

qrels columns: ['query-id', 'corpus-id', 'score', 'corpus-row']
Q_DEV: (6666, 384)
Corpus: (5416568, 384)
DEV qrels: 6666
FEVER DEV ALIGNMENT — PASS


## 4. Deterministic 50/50 boundary fit / validation split

In [9]:
def split_bucket(qid):
    h = hashlib.sha256(
        str(qid).encode("utf-8")
    ).digest()

    return int.from_bytes(
        h[:8],
        "big",
    ) % 2

boundary_split = pd.DataFrame({
    "query_id": dev_ids,
    "split": [
        "fit"
        if split_bucket(qid) == 0
        else "validation"
        for qid in dev_ids
    ],
})

boundary_split.to_csv(
    OUT / "v013_boundary_query_split.csv",
    index=False,
)

fit_mask = (
    boundary_split["split"].to_numpy()
    == "fit"
)

val_mask = ~fit_mask

print(boundary_split["split"].value_counts())
assert fit_mask.sum() + val_mask.sum() == 6666
print("FEVER BOUNDARY SPLIT — FROZEN")

split
fit           3350
validation    3316
Name: count, dtype: int64
FEVER BOUNDARY SPLIT — FROZEN


## 5. Retrieval, feedback, metric, and slope helpers

In [10]:
def cfg_key(cfg):
    method = cfg["method"]
    k = int(cfg["k"])
    alpha = str(cfg["alpha"]).replace(".", "p")

    if cfg["temperature"] is None:
        temp = "none"
    else:
        temp = str(cfg["temperature"]).replace(".", "p")

    return f"{method}-k{k}-a{alpha}-t{temp}"

def cosine_distance_rows(a, b):
    an = a / np.maximum(
        np.linalg.norm(a, axis=1, keepdims=True),
        1e-12,
    )
    bn = b / np.maximum(
        np.linalg.norm(b, axis=1, keepdims=True),
        1e-12,
    )
    return (
        1.0
        - np.sum(an * bn, axis=1)
    ).astype(np.float32)

def jaccard_distance_rows(a, b):
    out = np.empty(len(a), np.float32)

    for i in range(len(a)):
        A = set(map(int, a[i]))
        B = set(map(int, b[i]))
        union = len(A | B)
        out[i] = (
            1.0
            - len(A & B) / max(union, 1)
        )

    return out

def score_entropy(scores, k=20):
    z = np.asarray(
        scores[:, :k],
        np.float64,
    )

    z -= z.max(
        axis=1,
        keepdims=True,
    )

    p = np.exp(
        np.clip(z, -60, 60)
    )

    p /= np.maximum(
        p.sum(axis=1, keepdims=True),
        1e-12,
    )

    return -np.sum(
        p * np.log(np.maximum(p, 1e-12)),
        axis=1,
    )

def score_margin(scores):
    return np.asarray(
        scores[:, 0] - scores[:, 9],
        np.float32,
    )

def ndcg_at_10(qids, ids):
    discounts = (
        1.0
        / np.log2(np.arange(2, 12))
    )

    out = np.zeros(
        len(qids),
        np.float32,
    )

    for i, qid in enumerate(qids):
        rel = dev_qrels[str(qid)]

        hits = np.asarray(
            [
                int(doc) in rel
                for doc in ids[i, :10]
            ],
            dtype=np.float64,
        )

        dcg = float(
            (hits * discounts).sum()
        )

        ideal = min(len(rel), 10)

        idcg = float(
            discounts[:ideal].sum()
        )

        out[i] = (
            dcg / idcg
            if idcg
            else 0.0
        )

    return out

def fetch_doc_vectors(ids):
    flat = np.asarray(ids, np.int64).reshape(-1)

    if (flat < 0).any():
        raise ValueError("FAISS returned negative IDs")

    vec = np.asarray(
        CORPUS[flat],
        dtype=np.float32,
    )

    vec /= np.maximum(
        np.linalg.norm(vec, axis=1, keepdims=True),
        1e-12,
    )

    return vec.reshape(
        ids.shape[0],
        ids.shape[1],
        DIM,
    )

def feedback_matrix(ids, scores, cfg):
    k = int(cfg["k"])

    ids_k = ids[:, :k]
    scores_k = np.asarray(
        scores[:, :k],
        np.float64,
    )

    docs = fetch_doc_vectors(ids_k)

    if cfg["method"] == "mean":
        weights = np.full(
            (len(ids_k), k),
            1.0 / k,
            dtype=np.float64,
        )

    elif cfg["method"] == "softmax":
        T = float(cfg["temperature"])

        z = scores_k / T
        z -= z.max(axis=1, keepdims=True)

        weights = np.exp(
            np.clip(z, -60, 60)
        )

        weights /= np.maximum(
            weights.sum(axis=1, keepdims=True),
            1e-12,
        )

    else:
        raise ValueError(cfg["method"])

    fb = np.sum(
        docs * weights[:, :, None],
        axis=1,
    ).astype(np.float32)

    fb /= np.maximum(
        np.linalg.norm(fb, axis=1, keepdims=True),
        1e-12,
    )

    return fb

def anchored_update(q0, feedback, alpha):
    q = (
        (1.0 - float(alpha)) * q0
        + float(alpha) * feedback
    ).astype(np.float32)

    q /= np.maximum(
        np.linalg.norm(q, axis=1, keepdims=True),
        1e-12,
    )

    return q

def slopes_from_trajectory(df):
    rows = []

    group_cols = [
        "query_id",
        "low",
        "high",
        "method",
        "alpha",
        "k",
        "temperature",
        "config_key",
    ]

    for keys, g in df.groupby(
        group_cols,
        dropna=False,
        sort=False,
    ):
        (
            qid,
            low,
            high,
            method,
            alpha,
            k,
            temp,
            cfgk,
        ) = keys

        g = g.sort_values("iteration")
        x = g["iteration"].to_numpy(np.float64)

        def slope(col):
            return float(
                np.polyfit(
                    x,
                    g[col].to_numpy(np.float64),
                    1,
                )[0]
            )

        rows.append({
            "query_id": str(qid),
            "low": low,
            "high": high,
            "method": method,
            "alpha": float(alpha),
            "k": int(k),
            "temperature": temp,
            "config_key": cfgk,
            "H1_slope": slope("query_divergence"),
            "H2_slope": slope("candidate_increment"),
            "H3_slope": slope("abs_utility_gap"),
        })

    return pd.DataFrame(rows)

## 6. Baseline PQ32 / SQ8 retrieval and initial-state features

In [11]:
faiss.omp_set_num_threads(8)

baseline_cache = {}

for name, path in {
    "PQ32": PQ32_PATH,
    "SQ8": SQ8_PATH,
}.items():

    print("\n" + "=" * 80)
    print(name)

    index = faiss.read_index(str(path))
    index.nprobe = NPROBE

    t0 = time.perf_counter()

    scores, ids = index.search(
        np.ascontiguousarray(
            Q_DEV,
            np.float32,
        ),
        TOP_RETRIEVE,
    )

    dt = time.perf_counter() - t0

    baseline_cache[name] = {
        "scores": scores,
        "ids": ids,
        "ndcg": ndcg_at_10(dev_ids, ids),
        "entropy20": score_entropy(scores, 20),
        "margin1_10": score_margin(scores),
    }

    print("seconds :", dt)
    print(
        "nDCG@10:",
        float(
            baseline_cache[name]["ndcg"].mean()
        ),
    )

    del index
    gc.collect()

pq = baseline_cache["PQ32"]
sq = baseline_cache["SQ8"]

features_df = pd.DataFrame({
    "query_id": dev_ids,
    "initial_candidate_divergence":
        jaccard_distance_rows(
            pq["ids"],
            sq["ids"],
        ),
    "initial_abs_utility_gap":
        np.abs(
            sq["ndcg"] - pq["ndcg"]
        ),
    "pq32_entropy20":
        pq["entropy20"],
    "sq8_entropy20":
        sq["entropy20"],
    "pq32_margin1_10":
        pq["margin1_10"],
    "sq8_margin1_10":
        sq["margin1_10"],
    "entropy_gap":
        np.abs(
            pq["entropy20"] - sq["entropy20"]
        ),
    "margin_gap":
        np.abs(
            pq["margin1_10"] - sq["margin1_10"]
        ),
})

assert len(features_df) == 6666
assert features_df.isna().sum().sum() == 0

features_df.to_parquet(
    OUT / "v013_initial_query_features.parquet",
    index=False,
)

print("BASELINE FEATURE RECONSTRUCTION — PASS")
display(features_df.head())


PQ32
seconds : 10.623641981999981
nDCG@10: 0.14359115064144135

SQ8
seconds : 18.178068433000135
nDCG@10: 0.16908863186836243
BASELINE FEATURE RECONSTRUCTION — PASS


,query_id,initial_candidate_divergence,initial_abs_utility_gap,pq32_entropy20,sq8_entropy20,pq32_margin1_10,sq8_margin1_10,entropy_gap,margin_gap
0,100030,0.765432,1.0,2.995701,2.995637,0.029385,0.050698,0.000064,0.021313
1,100038,0.802395,0.0,2.995711,2.995702,0.017611,0.016831,0.000009,0.000780
2,100083,0.837209,0.0,2.995718,2.995625,0.012849,0.042437,0.000093,0.029588
3,100088,0.795181,0.0,2.995700,2.995646,0.031328,0.042491,0.000055,0.011163
4,100169,0.795181,0.0,2.995718,2.995708,0.014900,0.014154,0.000010,0.000746


## 7. Synchronized PQ32↔SQ8 trajectory runner

In [12]:
def run_pair_trajectory(cfg, query_mask):
    idx = np.flatnonzero(query_mask)

    qids = [
        dev_ids[i]
        for i in idx
    ]

    q0 = Q_DEV[idx].copy()

    low = faiss.read_index(str(PQ32_PATH))
    high = faiss.read_index(str(SQ8_PATH))

    low.nprobe = NPROBE
    high.nprobe = NPROBE

    qL = q0.copy()
    qH = q0.copy()

    frames = []
    base_cdiv = None

    for t in range(MAX_ROUNDS + 1):
        sL, idL = low.search(
            np.ascontiguousarray(
                qL,
                np.float32,
            ),
            TOP_RETRIEVE,
        )

        sH, idH = high.search(
            np.ascontiguousarray(
                qH,
                np.float32,
            ),
            TOP_RETRIEVE,
        )

        qdiv = cosine_distance_rows(
            qL,
            qH,
        )

        cdiv = jaccard_distance_rows(
            idL,
            idH,
        )

        if t == 0:
            base_cdiv = cdiv.copy()

        nL = ndcg_at_10(qids, idL)
        nH = ndcg_at_10(qids, idH)

        frames.append(
            pd.DataFrame({
                "query_id": qids,
                "iteration": t,
                "query_divergence": qdiv,
                "candidate_increment":
                    cdiv - base_cdiv,
                "abs_utility_gap":
                    np.abs(nH - nL),
            })
        )

        if t < MAX_ROUNDS:
            qL = anchored_update(
                q0,
                feedback_matrix(
                    idL,
                    sL,
                    cfg,
                ),
                cfg["alpha"],
            )

            qH = anchored_update(
                q0,
                feedback_matrix(
                    idH,
                    sH,
                    cfg,
                ),
                cfg["alpha"],
            )

    del low, high
    gc.collect()

    out_df = pd.concat(
        frames,
        ignore_index=True,
    )

    out_df["low"] = "PQ32"
    out_df["high"] = "SQ8"
    out_df["method"] = cfg["method"]
    out_df["alpha"] = cfg["alpha"]
    out_df["k"] = cfg["k"]
    out_df["temperature"] = cfg["temperature"]
    out_df["config_key"] = cfg_key(cfg)

    return out_df

## 8. FIT sweep

This is the expensive stage.

The notebook writes one parquet per configuration, so interrupted Colab runs can resume
without recomputing completed configurations.

In [16]:
fit_frames = []

for i, cfg in enumerate(
    BOUNDARY_CONFIGS,
    1,
):
    key = cfg_key(cfg)

    path = (
        OUT / f"fit-{key}.parquet"
    )

    print(
        f"[FIT {i:02d}/{len(BOUNDARY_CONFIGS)}]",
        key,
    )

    if path.is_file():
        df = pd.read_parquet(path)
        print("  CACHE HIT")
    else:
        df = run_pair_trajectory(
            cfg,
            fit_mask,
        )

        df.to_parquet(
            path,
            index=False,
        )

    fit_frames.append(df)

fit_boundary = pd.concat(
    fit_frames,
    ignore_index=True,
)

fit_slopes = slopes_from_trajectory(
    fit_boundary
)

assert (
    fit_slopes["query_id"].nunique()
    == int(fit_mask.sum())
)

assert (
    fit_slopes["config_key"].nunique()
    == 44
)

fit_slopes.to_parquet(
    OUT / "v013_fit_query_config_slopes.parquet",
    index=False,
)

print("FIT SLOPES:", fit_slopes.shape)
print("FEVER FIT SWEEP — COMPLETE")

[FIT 01/44] mean-k5-a0p1-tnone


KeyboardInterrupt: 

## 9. Fit-only first-order boundary models

In [22]:
from pathlib import Path
import numpy as np
import pandas as pd

RUN = Path(
    "/content/drive/MyDrive/"
    "rag-pq-checkpoints/arc-v0/"
    "fever-boundary-external-replication-v013/"
    "20260817-140640"
)

fit_files = sorted(RUN.glob("fit-*.parquet"))

print("FIT files:", len(fit_files))
assert len(fit_files) == 44

frames = []

for i, p in enumerate(fit_files, 1):
    df = pd.read_parquet(p)

    required = {
        "query_id",
        "iteration",
        "abs_utility_gap",
        "config_key",
    }

    assert required.issubset(df.columns), (
        p.name,
        df.columns.tolist(),
    )

    frames.append(df)

    if i in [1, 10, 20, 30, 40, 44]:
        print(
            f"[{i:02d}/44]",
            p.name,
            df.shape,
        )

fit_boundary = pd.concat(
    frames,
    ignore_index=True,
)

print("\ntrajectory rows:", fit_boundary.shape)


# ------------------------------------------------------------
# Reconstruct H3 slopes exactly from persisted trajectories
# ------------------------------------------------------------

group_cols = [
    "query_id",
    "low",
    "high",
    "method",
    "alpha",
    "k",
    "temperature",
    "config_key",
]

rows = []

for keys, g in fit_boundary.groupby(
    group_cols,
    dropna=False,
):
    g = g.sort_values("iteration")

    x = g["iteration"].to_numpy(np.float64)
    y = g["abs_utility_gap"].to_numpy(np.float64)

    h3 = float(
        np.polyfit(
            x,
            y,
            1,
        )[0]
    )

    row = dict(zip(group_cols, keys))
    row["H3_slope"] = h3
    rows.append(row)

fit_slopes = pd.DataFrame(rows)

print("slope rows:", fit_slopes.shape)
print(
    "unique queries:",
    fit_slopes["query_id"].nunique(),
)
print(
    "unique configs:",
    fit_slopes["config_key"].nunique(),
)

assert fit_slopes["config_key"].nunique() == 44


# ------------------------------------------------------------
# H3 diagnostics
# ------------------------------------------------------------

h3 = fit_slopes["H3_slope"].to_numpy(np.float64)

print("\n" + "=" * 80)
print("FEVER H3 DISTRIBUTION")
print("=" * 80)

print("N     :", len(h3))
print("min   :", float(h3.min()))
print("max   :", float(h3.max()))
print("mean  :", float(h3.mean()))
print("median:", float(np.median(h3)))

print("\nQUANTILES")

for q in [
    0.00,
    0.10,
    0.25,
    0.50,
    0.60,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
    0.95,
    0.975,
    0.99,
    1.00,
]:
    print(
        f"q={q:>5.3f}: "
        f"{np.quantile(h3, q):.12g}"
    )

print("\nSIGN / ZERO MASS")

print(
    "H3 < 0:",
    float((h3 < 0).mean()),
)

print(
    "H3 = 0 EXACT:",
    float((h3 == 0).mean()),
)

print(
    "|H3| <= 1e-12:",
    float(
        np.isclose(
            h3,
            0.0,
            atol=1e-12,
            rtol=0.0,
        ).mean()
    ),
)

print(
    "H3 > 0:",
    float((h3 > 0).mean()),
)


# ------------------------------------------------------------
# Original preregistered q75 threshold
# ------------------------------------------------------------

threshold = float(
    np.quantile(
        h3,
        0.75,
    )
)

print("\n" + "=" * 80)
print("75TH-PERCENTILE TARGET")
print("=" * 80)

print("threshold:", repr(threshold))

print(
    "prevalence >= threshold:",
    float((h3 >= threshold).mean()),
)

print(
    "prevalence > threshold:",
    float((h3 > threshold).mean()),
)

print(
    "exact ties at threshold:",
    int((h3 == threshold).sum()),
)

print(
    "tie fraction:",
    float((h3 == threshold).mean()),
)


# ------------------------------------------------------------
# Original ±0.002 regime
# ------------------------------------------------------------

EPS = 0.002

print("\n" + "=" * 80)
print("±0.002 REGIME")
print("=" * 80)

print(
    "reversal:",
    float((h3 < -EPS).mean()),
)

print(
    "stable/null:",
    float(
        (
            (h3 >= -EPS)
            & (h3 <= EPS)
        ).mean()
    ),
)

print(
    "amplifying:",
    float((h3 > EPS).mean()),
)


# ------------------------------------------------------------
# Most frequent exact slopes
# ------------------------------------------------------------

values, counts = np.unique(
    h3,
    return_counts=True,
)

common = (
    pd.DataFrame({
        "H3_slope": values,
        "count": counts,
    })
    .assign(
        fraction=lambda d:
            d["count"] / len(h3)
    )
    .sort_values(
        "count",
        ascending=False,
    )
    .head(15)
)

print("\nMOST COMMON H3 VALUES")
display(common)

FIT files: 44
[01/44] fit-mean-k20-a0p1-tnone.parquet (16750, 12)
[10/44] fit-mean-k50-a0p3-tnone.parquet (16750, 12)
[20/44] fit-softmax-k20-a0p3-t0p5.parquet (16750, 12)
[30/44] fit-softmax-k5-a0p1-t0p1.parquet (16750, 12)
[40/44] fit-softmax-k5-a0p5-t0p5.parquet (16750, 12)
[44/44] fit-softmax-k5-a0p7-t0p5.parquet (16750, 12)


/tmp/ipykernel_3583/1712186641.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  fit_boundary = pd.concat(



trajectory rows: (737000, 12)
slope rows: (147400, 9)
unique queries: 3350
unique configs: 44

FEVER H3 DISTRIBUTION
N     : 147400
min   : -0.3000000000000003
max   : 0.3000000000000001
mean  : 0.006860398314296791
median: 0.0

QUANTILES
q=0.000: -0.3
q=0.100: 0
q=0.250: 0
q=0.500: 0
q=0.600: 0
q=0.700: 0
q=0.750: 0
q=0.800: 0
q=0.850: 0
q=0.900: 0
q=0.950: 0.0738140463829
q=0.975: 0.131138422415
q=0.990: 0.2
q=1.000: 0.3

SIGN / ZERO MASS
H3 < 0: 0.03755088195386703
H3 = 0 EXACT: 0.882483039348711
|H3| <= 1e-12: 0.9005902306648575
H3 > 0: 0.07996607869742198

75TH-PERCENTILE TARGET
threshold: 0.0
prevalence >= threshold: 0.962449118046133
prevalence > threshold: 0.07996607869742198
exact ties at threshold: 130078
tie fraction: 0.882483039348711

±0.002 REGIME
reversal: 0.021648575305291722
stable/null: 0.9010990502035278
amplifying: 0.07725237449118046

MOST COMMON H3 VALUES


,H3_slope,count,fraction
1095,0.000000e+00,130078,0.882483
2215,7.381405e-02,996,0.006757
2910,2.000000e-01,855,0.005801
1041,-1.050792e-16,707,0.004796
1068,-2.016483e-17,591,0.004009
2493,1.107211e-01,491,0.003331
1100,2.990062e-18,286,0.001940
2421,1.000000e-01,259,0.001757
2564,1.226294e-01,227,0.001540
357,-7.381405e-02,220,0.001493


In [23]:
# ============================================================
# ARC-v0.13.1
# FEVER Zero-Mass / Regime-Shift Audit
#
# PURPOSE
# ------------------------------------------------------------
# 1. Preserve original ARC-v0.13 preregistration.
# 2. Diagnose the q75=0 target degeneracy.
# 3. Quantify sparse instability under the PRE-EXISTING
#    regime threshold |H3_slope| > 0.002.
# 4. Test whether the zero-mass / stable-null regime is
#    consistent across policy families and hyperparameters.
# 5. Use validation artifacts if already present.
#
# IMPORTANT
# ------------------------------------------------------------
# - NO retrieval rerun
# - NO threshold tuning
# - NO test access
# - NO replacement of the original q75 result
# - H3 > 0 analysis is diagnostic only
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json
import math

import numpy as np
import pandas as pd


# ============================================================
# 0. Frozen paths / constants
# ============================================================

ARC_ROOT = Path(
    "/content/drive/MyDrive/"
    "rag-pq-checkpoints/arc-v0"
)

V013_RUN = (
    ARC_ROOT
    / "fever-boundary-external-replication-v013"
    / "20260817-140640"
)

V0131_OUT = (
    ARC_ROOT
    / "fever-zero-mass-regime-audit-v0131"
    / datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
)

V0131_OUT.mkdir(
    parents=True,
    exist_ok=False,
)

EPS = 0.002
SEED = 20260816
BOOTSTRAP_REPS = 2000

assert V013_RUN.is_dir(), V013_RUN

print("V0.13 source:", V013_RUN)
print("V0.13.1 output:", V0131_OUT)


# ============================================================
# 1. Helpers
# ============================================================

def sha256_file(path, chunk_size=16 * 1024 * 1024):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            block = f.read(chunk_size)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def slopes_from_trajectory(df):

    group_cols = [
        "query_id",
        "low",
        "high",
        "method",
        "alpha",
        "k",
        "temperature",
        "config_key",
    ]

    rows = []

    for keys, g in df.groupby(
        group_cols,
        dropna=False,
        sort=False,
    ):

        g = g.sort_values(
            "iteration"
        )

        x = g[
            "iteration"
        ].to_numpy(
            np.float64
        )

        y = g[
            "abs_utility_gap"
        ].to_numpy(
            np.float64
        )

        assert len(x) >= 2

        h3 = float(
            np.polyfit(
                x,
                y,
                1,
            )[0]
        )

        row = dict(
            zip(
                group_cols,
                keys,
            )
        )

        row[
            "H3_slope"
        ] = h3

        rows.append(
            row
        )

    return pd.DataFrame(
        rows
    )


def add_regime_columns(df):

    out = df.copy()

    h3 = out[
        "H3_slope"
    ].to_numpy(
        np.float64
    )

    out[
        "exact_zero"
    ] = (
        h3 == 0.0
    )

    out[
        "near_zero_1e12"
    ] = np.isclose(
        h3,
        0.0,
        atol=1e-12,
        rtol=0.0,
    )

    out[
        "positive_H3"
    ] = (
        h3 > 0.0
    )

    out[
        "negative_H3"
    ] = (
        h3 < 0.0
    )

    out[
        "regime"
    ] = np.select(
        [
            h3 > EPS,
            h3 < -EPS,
        ],
        [
            "amplifying",
            "reversal",
        ],
        default="stable_or_null",
    )

    out[
        "is_amplifying"
    ] = (
        out["regime"]
        == "amplifying"
    )

    out[
        "is_reversal"
    ] = (
        out["regime"]
        == "reversal"
    )

    out[
        "is_stable_or_null"
    ] = (
        out["regime"]
        == "stable_or_null"
    )

    return out


def bootstrap_query_fraction(
    df,
    column,
    reps=BOOTSTRAP_REPS,
    seed=SEED,
):

    # Query-cluster bootstrap:
    # resample queries, retaining all configs belonging to each query.

    query_ids = (
        df["query_id"]
        .drop_duplicates()
        .astype(str)
        .to_numpy()
    )

    grouped = {
        str(qid): g[column].to_numpy(np.float64)
        for qid, g in df.groupby("query_id")
    }

    rng = np.random.default_rng(seed)

    estimates = np.empty(
        reps,
        dtype=np.float64,
    )

    for b in range(reps):

        sampled = rng.choice(
            query_ids,
            size=len(query_ids),
            replace=True,
        )

        total_sum = 0.0
        total_n = 0

        for qid in sampled:

            x = grouped[str(qid)]

            total_sum += float(
                x.sum()
            )

            total_n += len(x)

        estimates[b] = (
            total_sum
            / total_n
        )

    return {
        "estimate":
            float(
                df[column].mean()
            ),

        "ci_low":
            float(
                np.quantile(
                    estimates,
                    0.025,
                )
            ),

        "ci_high":
            float(
                np.quantile(
                    estimates,
                    0.975,
                )
            ),
    }


# ============================================================
# 2. Recover FIT slopes from persisted v0.13 trajectories
# ============================================================

fit_files = sorted(
    V013_RUN.glob(
        "fit-*.parquet"
    )
)

print()
print("FIT files:", len(fit_files))

assert len(fit_files) == 44, (
    "Expected 44 FIT checkpoints, got "
    f"{len(fit_files)}"
)

fit_frames = []

for i, path in enumerate(
    fit_files,
    1,
):

    df = pd.read_parquet(
        path
    )

    required = {
        "query_id",
        "iteration",
        "abs_utility_gap",
        "config_key",
    }

    assert required.issubset(
        df.columns
    ), (
        path.name,
        df.columns.tolist(),
    )

    fit_frames.append(
        df
    )

    if i in [
        1,
        10,
        20,
        30,
        40,
        44,
    ]:

        print(
            f"[{i:02d}/44]",
            path.name,
            df.shape,
        )


fit_boundary = pd.concat(
    fit_frames,
    ignore_index=True,
)

print(
    "FIT trajectory rows:",
    fit_boundary.shape,
)

fit_slopes = slopes_from_trajectory(
    fit_boundary
)

fit_slopes = add_regime_columns(
    fit_slopes
)

print(
    "FIT slope rows:",
    fit_slopes.shape,
)

print(
    "FIT unique queries:",
    fit_slopes[
        "query_id"
    ].nunique(),
)

print(
    "FIT configs:",
    fit_slopes[
        "config_key"
    ].nunique(),
)

assert (
    fit_slopes[
        "config_key"
    ].nunique()
    == 44
)


# ============================================================
# 3. Recover validation slopes IF already persisted
# ============================================================

validation_files = sorted(
    V013_RUN.glob(
        "validation-*.parquet"
    )
)

print()
print(
    "Validation checkpoint files:",
    len(validation_files),
)

val_slopes = None

if len(validation_files) == 44:

    print(
        "Complete validation checkpoints found."
    )

    val_frames = []

    for i, path in enumerate(
        validation_files,
        1,
    ):

        df = pd.read_parquet(
            path
        )

        val_frames.append(
            df
        )

        if i in [
            1,
            10,
            20,
            30,
            40,
            44,
        ]:

            print(
                f"[VAL {i:02d}/44]",
                path.name,
                df.shape,
            )

    val_boundary = pd.concat(
        val_frames,
        ignore_index=True,
    )

    val_slopes = slopes_from_trajectory(
        val_boundary
    )

    val_slopes = add_regime_columns(
        val_slopes
    )

    print(
        "Validation slope rows:",
        val_slopes.shape,
    )

    print(
        "Validation unique queries:",
        val_slopes[
            "query_id"
        ].nunique(),
    )

    print(
        "Validation configs:",
        val_slopes[
            "config_key"
        ].nunique(),
    )

elif len(validation_files) == 0:

    print(
        "No persisted validation checkpoints yet."
    )

    print(
        "FIT-only regime audit will proceed."
    )

else:

    print(
        "WARNING: partial validation checkpoint set:"
    )

    print(
        len(validation_files),
        "/ 44"
    )

    print(
        "Do NOT combine partial validation with FIT conclusions."
    )


# ============================================================
# 4. Preregistered q75 degeneracy audit
# ============================================================

h3_fit = (
    fit_slopes[
        "H3_slope"
    ]
    .to_numpy(
        np.float64
    )
)

q75_fit = float(
    np.quantile(
        h3_fit,
        0.75,
    )
)

q90_fit = float(
    np.quantile(
        h3_fit,
        0.90,
    )
)

q95_fit = float(
    np.quantile(
        h3_fit,
        0.95,
    )
)

q75_ge_prevalence = float(
    (
        h3_fit
        >= q75_fit
    ).mean()
)

q75_gt_prevalence = float(
    (
        h3_fit
        > q75_fit
    ).mean()
)

q75_tie_fraction = float(
    (
        h3_fit
        == q75_fit
    ).mean()
)


print()
print("=" * 80)
print("PREREGISTERED Q75 TARGET AUDIT")
print("=" * 80)

print(
    "FIT q75:",
    q75_fit,
)

print(
    "FIT q90:",
    q90_fit,
)

print(
    "FIT q95:",
    q95_fit,
)

print(
    "P(H3 >= q75):",
    q75_ge_prevalence,
)

print(
    "P(H3 > q75):",
    q75_gt_prevalence,
)

print(
    "tie fraction at q75:",
    q75_tie_fraction,
)

print(
    "exact-zero fraction:",
    float(
        fit_slopes[
            "exact_zero"
        ].mean()
    ),
)

print(
    "near-zero <=1e-12 fraction:",
    float(
        fit_slopes[
            "near_zero_1e12"
        ].mean()
    ),
)


q75_degenerate = bool(
    (q75_fit == 0.0)
    and
    (
        q75_tie_fraction
        >= 0.10
    )
)

print(
    "\nQ75 TARGET DEGENERATE:",
    q75_degenerate,
)

assert q75_degenerate, (
    "Expected to reproduce the observed FEVER "
    "zero-mass q75 degeneracy."
)


# ============================================================
# 5. Overall preregistered regime proportions
# ============================================================

regime_rows = []

for name, column in [
    (
        "amplifying",
        "is_amplifying",
    ),
    (
        "stable_or_null",
        "is_stable_or_null",
    ),
    (
        "reversal",
        "is_reversal",
    ),
    (
        "exact_zero",
        "exact_zero",
    ),
    (
        "positive_H3_diagnostic",
        "positive_H3",
    ),
]:

    stats = bootstrap_query_fraction(
        fit_slopes,
        column,
    )

    regime_rows.append({
        "split":
            "fit",

        "metric":
            name,

        **stats,
    })


if val_slopes is not None:

    for name, column in [
        (
            "amplifying",
            "is_amplifying",
        ),
        (
            "stable_or_null",
            "is_stable_or_null",
        ),
        (
            "reversal",
            "is_reversal",
        ),
        (
            "exact_zero",
            "exact_zero",
        ),
        (
            "positive_H3_diagnostic",
            "positive_H3",
        ),
    ]:

        stats = bootstrap_query_fraction(
            val_slopes,
            column,
            seed=SEED + 1,
        )

        regime_rows.append({
            "split":
                "validation",

            "metric":
                name,

            **stats,
        })


overall_regime = pd.DataFrame(
    regime_rows
)

print()
print("=" * 80)
print("OVERALL REGIME FRACTIONS")
print("=" * 80)

display(
    overall_regime
)


# ============================================================
# 6. Policy-family stability
# ============================================================

family_summary = (
    fit_slopes
    .groupby(
        "method",
        as_index=False,
    )
    .agg(
        observations=(
            "H3_slope",
            "size",
        ),
        unique_queries=(
            "query_id",
            "nunique",
        ),
        mean_H3=(
            "H3_slope",
            "mean",
        ),
        median_H3=(
            "H3_slope",
            "median",
        ),
        exact_zero_fraction=(
            "exact_zero",
            "mean",
        ),
        positive_H3_fraction=(
            "positive_H3",
            "mean",
        ),
        amplifying_fraction=(
            "is_amplifying",
            "mean",
        ),
        stable_or_null_fraction=(
            "is_stable_or_null",
            "mean",
        ),
        reversal_fraction=(
            "is_reversal",
            "mean",
        ),
    )
)

print()
print("=" * 80)
print("FIT POLICY-FAMILY REGIME SUMMARY")
print("=" * 80)

display(
    family_summary
)


# ============================================================
# 7. Hyperparameter regime map
# ============================================================

config_summary = (
    fit_slopes
    .groupby(
        [
            "method",
            "alpha",
            "k",
            "temperature",
            "config_key",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        observations=(
            "H3_slope",
            "size",
        ),
        mean_H3=(
            "H3_slope",
            "mean",
        ),
        median_H3=(
            "H3_slope",
            "median",
        ),
        exact_zero_fraction=(
            "exact_zero",
            "mean",
        ),
        positive_H3_fraction=(
            "positive_H3",
            "mean",
        ),
        amplifying_fraction=(
            "is_amplifying",
            "mean",
        ),
        stable_or_null_fraction=(
            "is_stable_or_null",
            "mean",
        ),
        reversal_fraction=(
            "is_reversal",
            "mean",
        ),
    )
)


print()
print("=" * 80)
print("MOST AMPLIFYING CONFIGS")
print("=" * 80)

display(
    config_summary
    .sort_values(
        "amplifying_fraction",
        ascending=False,
    )
    .head(15)
)


print()
print("=" * 80)
print("MOST STABLE/NULL CONFIGS")
print("=" * 80)

display(
    config_summary
    .sort_values(
        "stable_or_null_fraction",
        ascending=False,
    )
    .head(15)
)


# ============================================================
# 8. Distribution of amplification prevalence across configs
# ============================================================

amp = (
    config_summary[
        "amplifying_fraction"
    ]
    .to_numpy(
        np.float64
    )
)

stable = (
    config_summary[
        "stable_or_null_fraction"
    ]
    .to_numpy(
        np.float64
    )
)

print()
print("=" * 80)
print("CONFIG-LEVEL ROBUSTNESS")
print("=" * 80)

print(
    "Amplifying fraction:"
)

print(
    "  min   =",
    float(
        amp.min()
    ),
)

print(
    "  median=",
    float(
        np.median(
            amp
        )
    ),
)

print(
    "  max   =",
    float(
        amp.max()
    ),
)

print(
    "Stable/null fraction:"
)

print(
    "  min   =",
    float(
        stable.min()
    ),
)

print(
    "  median=",
    float(
        np.median(
            stable
        )
    ),
)

print(
    "  max   =",
    float(
        stable.max()
    ),
)


# ============================================================
# 9. Query-level susceptibility
#
# Fraction of configs under which a query becomes amplifying.
# ============================================================

query_summary = (
    fit_slopes
    .groupby(
        "query_id",
        as_index=False,
    )
    .agg(
        configs=(
            "config_key",
            "nunique",
        ),
        mean_H3=(
            "H3_slope",
            "mean",
        ),
        max_H3=(
            "H3_slope",
            "max",
        ),
        amplification_rate=(
            "is_amplifying",
            "mean",
        ),
        reversal_rate=(
            "is_reversal",
            "mean",
        ),
        exact_zero_rate=(
            "exact_zero",
            "mean",
        ),
    )
)

assert (
    query_summary[
        "configs"
    ]
    == 44
).all()


print()
print("=" * 80)
print("QUERY-LEVEL SUSCEPTIBILITY")
print("=" * 80)

print(
    "queries never amplifying:",
    float(
        (
            query_summary[
                "amplification_rate"
            ]
            == 0
        ).mean()
    ),
)

print(
    "queries amplifying under >=1 config:",
    float(
        (
            query_summary[
                "amplification_rate"
            ]
            > 0
        ).mean()
    ),
)

print(
    "queries amplifying under >=25% configs:",
    float(
        (
            query_summary[
                "amplification_rate"
            ]
            >= 0.25
        ).mean()
    ),
)

print(
    "queries amplifying under >=50% configs:",
    float(
        (
            query_summary[
                "amplification_rate"
            ]
            >= 0.50
        ).mean()
    ),
)


print()
print("Most susceptible queries:")

display(
    query_summary
    .sort_values(
        [
            "amplification_rate",
            "max_H3",
        ],
        ascending=False,
    )
    .head(20)
)


# ============================================================
# 10. Validation-vs-FIT regime stability if validation exists
# ============================================================

split_comparison = None

if val_slopes is not None:

    fit_split = (
        fit_slopes
        .groupby(
            "config_key",
            as_index=False,
        )
        .agg(
            fit_amplifying=(
                "is_amplifying",
                "mean",
            ),
            fit_stable=(
                "is_stable_or_null",
                "mean",
            ),
            fit_zero=(
                "exact_zero",
                "mean",
            ),
        )
    )

    val_split = (
        val_slopes
        .groupby(
            "config_key",
            as_index=False,
        )
        .agg(
            val_amplifying=(
                "is_amplifying",
                "mean",
            ),
            val_stable=(
                "is_stable_or_null",
                "mean",
            ),
            val_zero=(
                "exact_zero",
                "mean",
            ),
        )
    )

    split_comparison = fit_split.merge(
        val_split,
        on="config_key",
        how="inner",
        validate="one_to_one",
    )

    split_comparison[
        "amplifying_abs_gap"
    ] = np.abs(
        split_comparison[
            "fit_amplifying"
        ]
        -
        split_comparison[
            "val_amplifying"
        ]
    )

    split_comparison[
        "stable_abs_gap"
    ] = np.abs(
        split_comparison[
            "fit_stable"
        ]
        -
        split_comparison[
            "val_stable"
        ]
    )

    print()
    print("=" * 80)
    print("FIT ↔ VALIDATION REGIME STABILITY")
    print("=" * 80)

    print(
        "mean amplifying abs gap:",
        float(
            split_comparison[
                "amplifying_abs_gap"
            ].mean()
        ),
    )

    print(
        "max amplifying abs gap:",
        float(
            split_comparison[
                "amplifying_abs_gap"
            ].max()
        ),
    )

    display(
        split_comparison
        .sort_values(
            "amplifying_abs_gap",
            ascending=False,
        )
        .head(15)
    )


# ============================================================
# 11. Save audit artifacts
# ============================================================

fit_slopes.to_parquet(
    V0131_OUT
    / "v0131_fit_query_config_regimes.parquet",
    index=False,
)

overall_regime.to_csv(
    V0131_OUT
    / "v0131_overall_regime_summary.csv",
    index=False,
)

family_summary.to_csv(
    V0131_OUT
    / "v0131_policy_family_regime_summary.csv",
    index=False,
)

config_summary.to_csv(
    V0131_OUT
    / "v0131_config_regime_summary.csv",
    index=False,
)

query_summary.to_csv(
    V0131_OUT
    / "v0131_query_susceptibility_summary.csv",
    index=False,
)

if val_slopes is not None:

    val_slopes.to_parquet(
        V0131_OUT
        / "v0131_validation_query_config_regimes.parquet",
        index=False,
    )

if split_comparison is not None:

    split_comparison.to_csv(
        V0131_OUT
        / "v0131_fit_validation_regime_stability.csv",
        index=False,
    )


# ============================================================
# 12. Seal interpretation-safe audit report
# ============================================================

report = {
    "status":
        "ARC_V0131_FEVER_ZERO_MASS_REGIME_AUDIT_COMPLETE",

    "source_v013_run":
        str(
            V013_RUN
        ),

    "analysis_type":
        "posthoc_diagnostic_and_preregistered_regime_audit",

    "retrieval_rerun":
        False,

    "test_accessed":
        False,

    "regime_threshold_abs_slope":
        EPS,

    "regime_threshold_status":
        (
            "pre-existing ARC-v0.13 preregistered threshold; "
            "not tuned after observing FEVER"
        ),

    "fit_observations":
        int(
            len(
                fit_slopes
            )
        ),

    "fit_queries":
        int(
            fit_slopes[
                "query_id"
            ].nunique()
        ),

    "configs":
        int(
            fit_slopes[
                "config_key"
            ].nunique()
        ),

    "fit_q75_H3":
        q75_fit,

    "fit_q90_H3":
        q90_fit,

    "fit_q95_H3":
        q95_fit,

    "fit_q75_ge_prevalence":
        q75_ge_prevalence,

    "fit_q75_gt_prevalence":
        q75_gt_prevalence,

    "fit_q75_tie_fraction":
        q75_tie_fraction,

    "fit_exact_zero_fraction":
        float(
            fit_slopes[
                "exact_zero"
            ].mean()
        ),

    "fit_near_zero_1e12_fraction":
        float(
            fit_slopes[
                "near_zero_1e12"
            ].mean()
        ),

    "fit_amplifying_fraction":
        float(
            fit_slopes[
                "is_amplifying"
            ].mean()
        ),

    "fit_stable_or_null_fraction":
        float(
            fit_slopes[
                "is_stable_or_null"
            ].mean()
        ),

    "fit_reversal_fraction":
        float(
            fit_slopes[
                "is_reversal"
            ].mean()
        ),

    "q75_target_degenerate_due_to_zero_mass":
        q75_degenerate,

    "validation_checkpoints_found":
        int(
            len(
                validation_files
            )
        ),

    "validation_complete":
        bool(
            val_slopes
            is not None
        ),

    "interpretation_constraint":
        (
            "The original FIT-family q75 high-amplification "
            "binary target degenerates on FEVER because of a "
            "large point mass at H3_slope=0. The corresponding "
            "binary classifier AUC must not be interpreted as "
            "a valid high-amplification external-replication "
            "result. The pre-existing ±0.002 regime analysis "
            "remains interpretable."
        ),

    "completed_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


REPORT_PATH = (
    V0131_OUT
    / "v0131_zero_mass_regime_audit_report.json"
)

REPORT_PATH.write_text(
    json.dumps(
        report,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

report_sha = sha256_file(
    REPORT_PATH
)

(
    V0131_OUT
    / "V0131_REPORT_SHA256.txt"
).write_text(
    report_sha
    + "  "
    + REPORT_PATH.name
    + "\n",
    encoding="utf-8",
)


# ============================================================
# 13. Final decision output
# ============================================================

print()
print("=" * 80)
print("ARC-v0.13.1 ZERO-MASS / REGIME AUDIT — COMPLETE")
print("=" * 80)

print(
    "q75 target degenerate:",
    q75_degenerate,
)

print(
    "FIT exact zero:",
    float(
        fit_slopes[
            "exact_zero"
        ].mean()
    ),
)

print(
    "FIT stable/null:",
    float(
        fit_slopes[
            "is_stable_or_null"
        ].mean()
    ),
)

print(
    "FIT amplifying:",
    float(
        fit_slopes[
            "is_amplifying"
        ].mean()
    ),
)

print(
    "FIT reversal:",
    float(
        fit_slopes[
            "is_reversal"
        ].mean()
    ),
)

print(
    "Validation complete:",
    val_slopes is not None,
)

print(
    "Output:",
    V0131_OUT,
)

print(
    "Report SHA-256:",
    report_sha,
)

print("=" * 80)

V0.13 source: /content/drive/MyDrive/rag-pq-checkpoints/arc-v0/fever-boundary-external-replication-v013/20260817-140640
V0.13.1 output: /content/drive/MyDrive/rag-pq-checkpoints/arc-v0/fever-zero-mass-regime-audit-v0131/20260817-153420

FIT files: 44
[01/44] fit-mean-k20-a0p1-tnone.parquet (16750, 12)
[10/44] fit-mean-k50-a0p3-tnone.parquet (16750, 12)
[20/44] fit-softmax-k20-a0p3-t0p5.parquet (16750, 12)
[30/44] fit-softmax-k5-a0p1-t0p1.parquet (16750, 12)
[40/44] fit-softmax-k5-a0p5-t0p5.parquet (16750, 12)
[44/44] fit-softmax-k5-a0p7-t0p5.parquet (16750, 12)


/tmp/ipykernel_3583/1988145708.py:377: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  fit_boundary = pd.concat(


FIT trajectory rows: (737000, 12)
FIT slope rows: (147400, 17)
FIT unique queries: 3350
FIT configs: 44

Validation checkpoint files: 0
No persisted validation checkpoints yet.
FIT-only regime audit will proceed.

PREREGISTERED Q75 TARGET AUDIT
FIT q75: 0.0
FIT q90: 0.0
FIT q95: 0.07381404638290404
P(H3 >= q75): 0.962449118046133
P(H3 > q75): 0.07996607869742198
tie fraction at q75: 0.882483039348711
exact-zero fraction: 0.882483039348711
near-zero <=1e-12 fraction: 0.9005902306648575

Q75 TARGET DEGENERATE: True

OVERALL REGIME FRACTIONS


,split,metric,estimate,ci_low,ci_high
0,fit,amplifying,0.077252,0.070420,0.083965
1,fit,stable_or_null,0.901099,0.893155,0.909697
2,fit,reversal,0.021649,0.018377,0.024939
3,fit,exact_zero,0.882483,0.873175,0.892660
4,fit,positive_H3_diagnostic,0.079966,0.073039,0.086943



FIT POLICY-FAMILY REGIME SUMMARY


...


MOST STABLE/NULL CONFIGS


,method,alpha,k,temperature,config_key,observations,mean_H3,median_H3,exact_zero_fraction,positive_H3_fraction,amplifying_fraction,stable_or_null_fraction,reversal_fraction
14,softmax,0.1,5,0.20,softmax-k5-a0p1-t0p2,3350,0.001845,0.0,0.924179,0.037612,0.032836,0.961194,0.005970
12,softmax,0.1,5,0.05,softmax-k5-a0p1-t0p05,3350,0.001951,0.0,0.924179,0.038209,0.033433,0.961194,0.005373
0,mean,0.1,5,NaN,mean-k5-a0p1-tnone,3350,0.001790,0.0,0.924478,0.037612,0.032836,0.960896,0.006269
13,softmax,0.1,5,0.10,softmax-k5-a0p1-t0p1,3350,0.001877,0.0,0.924478,0.038209,0.033433,0.960896,0.005672
15,softmax,0.1,5,0.50,softmax-k5-a0p1-t0p5,3350,0.001793,0.0,0.924478,0.037612,0.032836,0.960896,0.006269
1,mean,0.1,20,NaN,mean-k20-a0p1-tnone,3350,0.001736,0.0,0.924776,0.038507,0.034030,0.959104,0.006866
19,softmax,0.1,20,0.50,softmax-k20-a0p1-t0p5,3350,0.001709,0.0,0.924776,0.038507,0.034030,0.959104,0.006866
16,softmax,0.1,20,0.05,softmax-k20-a0p1-t0p05,3350,0.001939,0.0,0.924179,0.039403,0.034925,0.958507,0.006567
18,softmax,0.1,20,0.20,softmax-k20-a0p1-t0p2,3350,0.001778,0.0,0.924776,0.039104,0.034627,0.958507,0.006866
17,softmax,0.1,20,0.10,softmax-k20-a0p1-t0p1,3350,0.001854,0.0,0.924179,0.039701,0.035224,0.957612,0.007164



CONFIG-LEVEL ROBUSTNESS
Amplifying fraction:
  min   = 0.03283582089552239
  median= 0.0791044776119403
  max   = 0.12417910447761193
Stable/null fraction:
  min   = 0.8328358208955224
  median= 0.9020895522388059
  max   = 0.9611940298507463

QUERY-LEVEL SUSCEPTIBILITY
queries never amplifying: 0.8241791044776119
queries amplifying under >=1 config: 0.17582089552238805
queries amplifying under >=25% configs: 0.12268656716417911
queries amplifying under >=50% configs: 0.07313432835820896

Most susceptible queries:


,query_id,configs,mean_H3,max_H3,amplification_rate,reversal_rate,exact_zero_rate
147,109834,44,0.177890,0.264379,1.000000,0.000000,0.000000
3181,88473,44,0.132707,0.238685,1.000000,0.000000,0.000000
378,124805,44,0.113154,0.233333,1.000000,0.000000,0.000000
1288,179738,44,0.152907,0.233333,1.000000,0.000000,0.000000
2916,70602,44,0.120330,0.230103,1.000000,0.000000,0.000000
2739,58800,44,0.078984,0.145312,1.000000,0.000000,0.000000
3273,94958,44,0.069504,0.143068,1.000000,0.000000,0.000000
2479,43666,44,0.082107,0.140073,1.000000,0.000000,0.000000
867,155722,44,0.067118,0.133333,1.000000,0.000000,0.000000
1716,202445,44,0.057440,0.124821,1.000000,0.000000,0.000000



ARC-v0.13.1 ZERO-MASS / REGIME AUDIT — COMPLETE
q75 target degenerate: True
FIT exact zero: 0.882483039348711
FIT stable/null: 0.9010990502035278
FIT amplifying: 0.07725237449118046
FIT reversal: 0.021648575305291722
Validation complete: False
Output: /content/drive/MyDrive/rag-pq-checkpoints/arc-v0/fever-zero-mass-regime-audit-v0131/20260817-153420
Report SHA-256: e05f3963be507b75246794f9ea342bf7b05b6d818a7b9819606e236aeb7003c9


In [29]:
# ============================================================
# ARC-v0.13
# Validation-only resume-safe continuation
#
# IMPORTANT
# ------------------------------------------------------------
# - ONLY runs untouched FEVER validation half
# - DOES NOT rerun FIT
# - DOES NOT touch test
# - DOES NOT change preregistered thresholds
# - Saves every config immediately to Google Drive
# - Resume-safe after disconnect
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib
import time
import gc

import numpy as np
import pandas as pd


# ============================================================
# 0. Frozen source run
# ============================================================

ARC_ROOT = Path(
    "/content/drive/MyDrive/"
    "rag-pq-checkpoints/arc-v0"
)

V013_RUN = (
    ARC_ROOT
    / "fever-boundary-external-replication-v013"
    / "20260817-140640"
)

assert V013_RUN.is_dir(), V013_RUN

print("V0.13 frozen run:", V013_RUN)


# ============================================================
# 1. Required frozen runtime objects
# ============================================================

required_objects = [
    "BOUNDARY_CONFIGS",
    "run_pair_trajectory",
    "slopes_from_trajectory",
    "cfg_key",
    "val_mask",
    "dev_ids",
    "MAX_ROUNDS",
]

missing = [
    name
    for name in required_objects
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing required v0.13 runtime objects:\n"
        + "\n".join(
            f"  - {x}"
            for x in missing
        )
        + "\n\n"
        "Run only the original v0.13 setup/helper cells. "
        "Do NOT rerun FIT."
    )

assert len(BOUNDARY_CONFIGS) == 44

expected_val_queries = int(
    np.sum(val_mask)
)

assert expected_val_queries == 3316

print("Boundary configs:", len(BOUNDARY_CONFIGS))
print("Validation queries:", expected_val_queries)


# ============================================================
# 2. Verify function signature
# ============================================================

import inspect

sig = inspect.signature(
    run_pair_trajectory
)

print(
    "run_pair_trajectory signature:",
    sig,
)

assert list(sig.parameters.keys()) == [
    "cfg",
    "query_mask",
], sig

print("FUNCTION SIGNATURE — PASS")


# ============================================================
# 3. Verify sealed protocol
# ============================================================

PROTOCOL_PATH = (
    V013_RUN
    / "v013_fever_boundary_protocol.json"
)

assert PROTOCOL_PATH.is_file(), PROTOCOL_PATH

with open(
    PROTOCOL_PATH,
    "r",
    encoding="utf-8",
) as f:
    protocol = json.load(f)

print(
    "Protocol status:",
    protocol["status"],
)

assert (
    protocol["status"]
    == "FEVER_BOUNDARY_EXTERNAL_REPLICATION_SEALED_BEFORE_SWEEP"
)

assert protocol["test_access_allowed"] is False
assert protocol["boundary_grid_config_count"] == 44

print("PROTOCOL CHECK — PASS")


# ============================================================
# 4. SHA helper
# ============================================================

def sha256_file(
    path,
    chunk_size=16 * 1024 * 1024,
):
    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            block = f.read(
                chunk_size
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


# ============================================================
# 5. Checkpoint validator
# ============================================================

def validate_checkpoint(
    path,
    expected_queries,
):

    try:
        df = pd.read_parquet(
            path
        )

    except Exception as e:

        return (
            False,
            f"read failed: {repr(e)}",
        )

    required_cols = {
        "query_id",
        "iteration",
        "abs_utility_gap",
        "config_key",
    }

    if not required_cols.issubset(
        df.columns
    ):
        return (
            False,
            "required columns missing",
        )

    query_count = int(
        df[
            "query_id"
        ].nunique()
    )

    if query_count != expected_queries:

        return (
            False,
            (
                f"query count "
                f"{query_count} "
                f"!= {expected_queries}"
            ),
        )

    expected_rows = (
        expected_queries
        * (MAX_ROUNDS + 1)
    )

    if len(df) != expected_rows:

        return (
            False,
            (
                f"rows {len(df)} "
                f"!= {expected_rows}"
            ),
        )

    if (
        df["config_key"]
        .nunique()
        != 1
    ):
        return (
            False,
            "multiple config_key values",
        )

    return True, "ok"


# ============================================================
# 6. Audit existing checkpoints
# ============================================================

existing = sorted(
    V013_RUN.glob(
        "validation-*.parquet"
    )
)

print()
print("=" * 80)
print("EXISTING VALIDATION CHECKPOINTS")
print("=" * 80)

print(
    "Found:",
    len(existing),
)

valid_existing = []
invalid_existing = []

for path in existing:

    ok, reason = validate_checkpoint(
        path,
        expected_val_queries,
    )

    if ok:

        valid_existing.append(
            path
        )

    else:

        invalid_existing.append(
            (
                path,
                reason,
            )
        )

print(
    "Valid checkpoints:",
    len(valid_existing),
)

print(
    "Invalid checkpoints:",
    len(invalid_existing),
)

if invalid_existing:

    for path, reason in invalid_existing:

        print(
            "INVALID:",
            path.name,
            reason,
        )

    raise RuntimeError(
        "Invalid existing validation checkpoint detected. "
        "Stopping without overwrite."
    )


# ============================================================
# 7. Run ONLY missing validation configs
# ============================================================

manifest_rows = []

overall_start = time.perf_counter()

for i, cfg in enumerate(
    BOUNDARY_CONFIGS,
    1,
):

    key = cfg_key(
        cfg
    )

    out_path = (
        V013_RUN
        / f"validation-{key}.parquet"
    )

    print()
    print("=" * 80)
    print(
        f"[VAL {i:02d}/44]",
        key,
    )

    # --------------------------------------------------------
    # Resume-safe skip
    # --------------------------------------------------------

    if out_path.is_file():

        ok, reason = validate_checkpoint(
            out_path,
            expected_val_queries,
        )

        if not ok:

            raise RuntimeError(
                f"Existing checkpoint invalid:\n"
                f"{out_path}\n"
                f"{reason}"
            )

        existing_df = pd.read_parquet(
            out_path
        )

        print(
            "SKIP — valid checkpoint already exists"
        )

        print(
            "rows:",
            len(existing_df),
        )

        print(
            "queries:",
            existing_df[
                "query_id"
            ].nunique(),
        )

        file_sha = sha256_file(
            out_path
        )

        print(
            "sha256:",
            file_sha,
        )

        manifest_rows.append({
            "config_key":
                key,

            "status":
                "skipped_existing",

            "rows":
                int(
                    len(
                        existing_df
                    )
                ),

            "queries":
                int(
                    existing_df[
                        "query_id"
                    ].nunique()
                ),

            "seconds":
                0.0,

            "sha256":
                file_sha,

            "path":
                str(
                    out_path
                ),
        })

        del existing_df
        gc.collect()

        continue


    # --------------------------------------------------------
    # Compute validation trajectory
    #
    # IMPORTANT:
    # exact v0.13 signature is:
    #
    # run_pair_trajectory(cfg, query_mask)
    # --------------------------------------------------------

    t0 = time.perf_counter()

    df = run_pair_trajectory(
        cfg=cfg,
        query_mask=val_mask,
    )

    elapsed = (
        time.perf_counter()
        - t0
    )


    # --------------------------------------------------------
    # Strong pre-write checks
    # --------------------------------------------------------

    expected_rows = (
        expected_val_queries
        * (MAX_ROUNDS + 1)
    )

    assert len(df) == expected_rows, (
        key,
        len(df),
        expected_rows,
    )

    assert (
        df[
            "query_id"
        ].nunique()
        == expected_val_queries
    )

    assert (
        df[
            "config_key"
        ].nunique()
        == 1
    )

    assert (
        df[
            "config_key"
        ].iloc[0]
        == key
    )


    # --------------------------------------------------------
    # Temporary write then validation
    # --------------------------------------------------------

    tmp_path = Path(
        str(out_path)
        + ".partial"
    )

    if tmp_path.exists():
        tmp_path.unlink()

    df.to_parquet(
        tmp_path,
        index=False,
    )

    tmp_ok, tmp_reason = validate_checkpoint(
        tmp_path,
        expected_val_queries,
    )

    if not tmp_ok:

        raise RuntimeError(
            "Temporary checkpoint validation failed:\n"
            f"{tmp_path}\n"
            f"{tmp_reason}"
        )

    tmp_path.replace(
        out_path
    )


    # --------------------------------------------------------
    # Final persisted verification
    # --------------------------------------------------------

    final_ok, final_reason = validate_checkpoint(
        out_path,
        expected_val_queries,
    )

    if not final_ok:

        raise RuntimeError(
            "Final checkpoint validation failed:\n"
            f"{out_path}\n"
            f"{final_reason}"
        )

    file_sha = sha256_file(
        out_path
    )

    print(
        "SAVED:",
        out_path.name,
    )

    print(
        "seconds:",
        elapsed,
    )

    print(
        "rows:",
        len(df),
    )

    print(
        "queries:",
        df[
            "query_id"
        ].nunique(),
    )

    print(
        "sha256:",
        file_sha,
    )

    manifest_rows.append({
        "config_key":
            key,

        "status":
            "computed",

        "rows":
            int(
                len(df)
            ),

        "queries":
            int(
                df[
                    "query_id"
                ].nunique()
            ),

        "seconds":
            float(
                elapsed
            ),

        "sha256":
            file_sha,

        "path":
            str(
                out_path
            ),
    })

    del df
    gc.collect()


# ============================================================
# 8. Final checkpoint audit
# ============================================================

validation_files = sorted(
    V013_RUN.glob(
        "validation-*.parquet"
    )
)

print()
print("=" * 80)
print("FINAL VALIDATION CHECKPOINT AUDIT")
print("=" * 80)

print(
    "checkpoint files:",
    len(validation_files),
)

assert len(validation_files) == 44

for i, path in enumerate(
    validation_files,
    1,
):

    ok, reason = validate_checkpoint(
        path,
        expected_val_queries,
    )

    if not ok:

        raise RuntimeError(
            f"{path.name}: "
            f"{reason}"
        )

    if i in [
        1,
        10,
        20,
        30,
        40,
        44,
    ]:

        print(
            f"[{i:02d}/44]",
            path.name,
            "PASS",
        )

print(
    "\nVALIDATION CHECKPOINT SET — PASS"
)


# ============================================================
# 9. Load all validation trajectories
# ============================================================

val_frames = []

for i, path in enumerate(
    validation_files,
    1,
):

    df = pd.read_parquet(
        path
    )

    val_frames.append(
        df
    )

    if i in [
        1,
        10,
        20,
        30,
        40,
        44,
    ]:

        print(
            f"[LOAD {i:02d}/44]",
            path.name,
            df.shape,
        )

val_boundary = pd.concat(
    val_frames,
    ignore_index=True,
)

print(
    "\nValidation trajectory rows:",
    val_boundary.shape,
)

expected_total_rows = (
    expected_val_queries
    * 44
    * (MAX_ROUNDS + 1)
)

assert (
    len(val_boundary)
    == expected_total_rows
)

print(
    "Expected trajectory rows:",
    expected_total_rows,
)


# ============================================================
# 10. Reconstruct validation slopes
# ============================================================

val_slopes = slopes_from_trajectory(
    val_boundary
)

print(
    "Validation slope rows:",
    val_slopes.shape,
)

print(
    "Validation unique queries:",
    val_slopes[
        "query_id"
    ].nunique(),
)

print(
    "Validation configs:",
    val_slopes[
        "config_key"
    ].nunique(),
)

assert (
    val_slopes[
        "query_id"
    ].nunique()
    == expected_val_queries
)

assert (
    val_slopes[
        "config_key"
    ].nunique()
    == 44
)

assert (
    len(
        val_slopes
    )
    == expected_val_queries
    * 44
)

print(
    "VALIDATION SLOPE RECONSTRUCTION — PASS"
)


# ============================================================
# 11. Frozen preregistered regime analysis
# ============================================================

EPS = float(
    protocol[
        "regime_threshold_abs_slope"
    ]
)

assert np.isclose(
    EPS,
    0.002,
)

h3 = (
    val_slopes[
        "H3_slope"
    ]
    .to_numpy(
        np.float64
    )
)

val_slopes[
    "exact_zero"
] = (
    h3 == 0.0
)

val_slopes[
    "near_zero_1e12"
] = np.isclose(
    h3,
    0.0,
    atol=1e-12,
    rtol=0.0,
)

val_slopes[
    "positive_H3"
] = (
    h3 > 0.0
)

val_slopes[
    "negative_H3"
] = (
    h3 < 0.0
)

val_slopes[
    "is_amplifying"
] = (
    h3 > EPS
)

val_slopes[
    "is_reversal"
] = (
    h3 < -EPS
)

val_slopes[
    "is_stable_or_null"
] = (
    (~val_slopes["is_amplifying"])
    &
    (~val_slopes["is_reversal"])
)


# ============================================================
# 12. Untouched validation headline results
# ============================================================

q75_val = float(
    np.quantile(
        h3,
        0.75,
    )
)

q90_val = float(
    np.quantile(
        h3,
        0.90,
    )
)

q95_val = float(
    np.quantile(
        h3,
        0.95,
    )
)

q975_val = float(
    np.quantile(
        h3,
        0.975,
    )
)

print()
print("=" * 80)
print("UNTOUCHED FEVER VALIDATION RESULTS")
print("=" * 80)

print(
    "N:",
    len(h3),
)

print(
    "mean H3:",
    float(
        h3.mean()
    ),
)

print(
    "median H3:",
    float(
        np.median(
            h3
        )
    ),
)

print(
    "q75:",
    q75_val,
)

print(
    "q90:",
    q90_val,
)

print(
    "q95:",
    q95_val,
)

print(
    "q97.5:",
    q975_val,
)

print()

print(
    "exact zero:",
    float(
        val_slopes[
            "exact_zero"
        ].mean()
    ),
)

print(
    "near zero <=1e-12:",
    float(
        val_slopes[
            "near_zero_1e12"
        ].mean()
    ),
)

print(
    "positive H3:",
    float(
        val_slopes[
            "positive_H3"
        ].mean()
    ),
)

print(
    "negative H3:",
    float(
        val_slopes[
            "negative_H3"
        ].mean()
    ),
)

print()

print(
    "stable/null:",
    float(
        val_slopes[
            "is_stable_or_null"
        ].mean()
    ),
)

print(
    "amplifying:",
    float(
        val_slopes[
            "is_amplifying"
        ].mean()
    ),
)

print(
    "reversal:",
    float(
        val_slopes[
            "is_reversal"
        ].mean()
    ),
)


# ============================================================
# 13. Policy-family summary
# ============================================================

family_summary_val = (
    val_slopes
    .groupby(
        "method",
        as_index=False,
    )
    .agg(
        observations=(
            "H3_slope",
            "size",
        ),

        unique_queries=(
            "query_id",
            "nunique",
        ),

        mean_H3=(
            "H3_slope",
            "mean",
        ),

        median_H3=(
            "H3_slope",
            "median",
        ),

        exact_zero_fraction=(
            "exact_zero",
            "mean",
        ),

        positive_H3_fraction=(
            "positive_H3",
            "mean",
        ),

        amplifying_fraction=(
            "is_amplifying",
            "mean",
        ),

        stable_or_null_fraction=(
            "is_stable_or_null",
            "mean",
        ),

        reversal_fraction=(
            "is_reversal",
            "mean",
        ),
    )
)

print()
print("=" * 80)
print("VALIDATION POLICY-FAMILY SUMMARY")
print("=" * 80)

display(
    family_summary_val
)


# ============================================================
# 14. Config-level validation summary
# ============================================================

config_summary_val = (
    val_slopes
    .groupby(
        [
            "method",
            "alpha",
            "k",
            "temperature",
            "config_key",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        observations=(
            "H3_slope",
            "size",
        ),

        mean_H3=(
            "H3_slope",
            "mean",
        ),

        median_H3=(
            "H3_slope",
            "median",
        ),

        exact_zero_fraction=(
            "exact_zero",
            "mean",
        ),

        positive_H3_fraction=(
            "positive_H3",
            "mean",
        ),

        amplifying_fraction=(
            "is_amplifying",
            "mean",
        ),

        stable_or_null_fraction=(
            "is_stable_or_null",
            "mean",
        ),

        reversal_fraction=(
            "is_reversal",
            "mean",
        ),
    )
)

print()
print("=" * 80)
print("MOST AMPLIFYING VALIDATION CONFIGS")
print("=" * 80)

display(
    config_summary_val
    .sort_values(
        "amplifying_fraction",
        ascending=False,
    )
    .head(15)
)

print()
print("=" * 80)
print("MOST STABLE VALIDATION CONFIGS")
print("=" * 80)

display(
    config_summary_val
    .sort_values(
        "stable_or_null_fraction",
        ascending=False,
    )
    .head(15)
)


# ============================================================
# 15. Query-level susceptibility
# ============================================================

query_summary_val = (
    val_slopes
    .groupby(
        "query_id",
        as_index=False,
    )
    .agg(
        configs=(
            "config_key",
            "nunique",
        ),

        mean_H3=(
            "H3_slope",
            "mean",
        ),

        max_H3=(
            "H3_slope",
            "max",
        ),

        amplification_rate=(
            "is_amplifying",
            "mean",
        ),

        reversal_rate=(
            "is_reversal",
            "mean",
        ),

        exact_zero_rate=(
            "exact_zero",
            "mean",
        ),
    )
)

assert (
    query_summary_val[
        "configs"
    ]
    == 44
).all()

never_amp = float(
    (
        query_summary_val[
            "amplification_rate"
        ]
        == 0
    ).mean()
)

any_amp = float(
    (
        query_summary_val[
            "amplification_rate"
        ]
        > 0
    ).mean()
)

amp_25 = float(
    (
        query_summary_val[
            "amplification_rate"
        ]
        >= 0.25
    ).mean()
)

amp_50 = float(
    (
        query_summary_val[
            "amplification_rate"
        ]
        >= 0.50
    ).mean()
)

print()
print("=" * 80)
print("VALIDATION QUERY SUSCEPTIBILITY")
print("=" * 80)

print(
    "never amplifying:",
    never_amp,
)

print(
    ">=1 amplifying config:",
    any_amp,
)

print(
    ">=25% configs amplifying:",
    amp_25,
)

print(
    ">=50% configs amplifying:",
    amp_50,
)

print()

display(
    query_summary_val
    .sort_values(
        [
            "amplification_rate",
            "max_H3",
        ],
        ascending=False,
    )
    .head(20)
)


# ============================================================
# 16. Save validation aggregate artifacts
# ============================================================

VAL_SLOPES_PATH = (
    V013_RUN
    / "v013_validation_query_config_slopes.parquet"
)

FAMILY_PATH = (
    V013_RUN
    / "v013_validation_policy_family_summary.csv"
)

CONFIG_PATH = (
    V013_RUN
    / "v013_validation_config_regime_summary.csv"
)

QUERY_PATH = (
    V013_RUN
    / "v013_validation_query_susceptibility.csv"
)

MANIFEST_PATH = (
    V013_RUN
    / "v013_validation_checkpoint_manifest.csv"
)

val_slopes.to_parquet(
    VAL_SLOPES_PATH,
    index=False,
)

family_summary_val.to_csv(
    FAMILY_PATH,
    index=False,
)

config_summary_val.to_csv(
    CONFIG_PATH,
    index=False,
)

query_summary_val.to_csv(
    QUERY_PATH,
    index=False,
)

manifest_df = pd.DataFrame(
    manifest_rows
)

manifest_df.to_csv(
    MANIFEST_PATH,
    index=False,
)


# ============================================================
# 17. Seal validation continuation report
# ============================================================

validation_report = {
    "status":
        "ARC_V013_FEVER_VALIDATION_CONTINUATION_COMPLETE",

    "source_run":
        str(
            V013_RUN
        ),

    "protocol_sha256":
        protocol[
            "protocol_sha256"
        ],

    "validation_queries":
        int(
            expected_val_queries
        ),

    "validation_configs":
        44,

    "validation_observations":
        int(
            len(
                val_slopes
            )
        ),

    "q75_H3":
        q75_val,

    "q90_H3":
        q90_val,

    "q95_H3":
        q95_val,

    "q975_H3":
        q975_val,

    "exact_zero_fraction":
        float(
            val_slopes[
                "exact_zero"
            ].mean()
        ),

    "near_zero_1e12_fraction":
        float(
            val_slopes[
                "near_zero_1e12"
            ].mean()
        ),

    "stable_or_null_fraction":
        float(
            val_slopes[
                "is_stable_or_null"
            ].mean()
        ),

    "amplifying_fraction":
        float(
            val_slopes[
                "is_amplifying"
            ].mean()
        ),

    "reversal_fraction":
        float(
            val_slopes[
                "is_reversal"
            ].mean()
        ),

    "positive_H3_diagnostic_fraction":
        float(
            val_slopes[
                "positive_H3"
            ].mean()
        ),

    "query_never_amplifying_fraction":
        never_amp,

    "query_any_amplifying_fraction":
        any_amp,

    "query_amplifying_ge25pct_configs_fraction":
        amp_25,

    "query_amplifying_ge50pct_configs_fraction":
        amp_50,

    "fit_rerun":
        False,

    "test_accessed":
        False,

    "threshold_tuning_after_fit":
        False,

    "validation_checkpoint_count":
        44,

    "completed_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}


VALIDATION_REPORT_PATH = (
    V013_RUN
    / "v013_validation_continuation_report.json"
)

VALIDATION_REPORT_PATH.write_text(
    json.dumps(
        validation_report,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

validation_report_sha = sha256_file(
    VALIDATION_REPORT_PATH
)

(
    V013_RUN
    / "V013_VALIDATION_REPORT_SHA256.txt"
).write_text(
    validation_report_sha
    + "  "
    + VALIDATION_REPORT_PATH.name
    + "\n",
    encoding="utf-8",
)


# ============================================================
# 18. Final output
# ============================================================

total_elapsed = (
    time.perf_counter()
    - overall_start
)

print()
print("=" * 80)
print("ARC-v0.13 VALIDATION-ONLY CONTINUATION — PASS")
print("=" * 80)

print(
    "Validation checkpoints:",
    len(
        validation_files
    ),
)

print(
    "Validation queries:",
    expected_val_queries,
)

print(
    "Validation configs:",
    val_slopes[
        "config_key"
    ].nunique(),
)

print(
    "Exact zero:",
    float(
        val_slopes[
            "exact_zero"
        ].mean()
    ),
)

print(
    "Stable/null:",
    float(
        val_slopes[
            "is_stable_or_null"
        ].mean()
    ),
)

print(
    "Amplifying:",
    float(
        val_slopes[
            "is_amplifying"
        ].mean()
    ),
)

print(
    "Reversal:",
    float(
        val_slopes[
            "is_reversal"
        ].mean()
    ),
)

print(
    "Total continuation seconds:",
    total_elapsed,
)

print(
    "Validation report:",
    VALIDATION_REPORT_PATH,
)

print(
    "Report SHA-256:",
    validation_report_sha,
)

print("=" * 80)

V0.13 frozen run: /content/drive/MyDrive/rag-pq-checkpoints/arc-v0/fever-boundary-external-replication-v013/20260817-140640
Boundary configs: 44
Validation queries: 3316
run_pair_trajectory signature: (cfg, query_mask)
FUNCTION SIGNATURE — PASS
Protocol status: FEVER_BOUNDARY_EXTERNAL_REPLICATION_SEALED_BEFORE_SWEEP
PROTOCOL CHECK — PASS

EXISTING VALIDATION CHECKPOINTS
Found: 0
Valid checkpoints: 0
Invalid checkpoints: 0

[VAL 01/44] mean-k5-a0p1-tnone
SAVED: validation-mean-k5-a0p1-tnone.parquet
seconds: 88.38493943799995
rows: 16580
queries: 3316
sha256: 242c4281b632f954eb43769fa99b5966d770162c1e43a635b7a8d54c2e82d9b9

[VAL 02/44] mean-k20-a0p1-tnone
SAVED: validation-mean-k20-a0p1-tnone.parquet
seconds: 84.34124216999999
rows: 16580
queries: 3316
sha256: 35346fd4320f78593b82fb28ce99e4e989b071dcac94d961fcd77329778a93fc

[VAL 03/44] mean-k50-a0p1-tnone
SAVED: validation-mean-k50-a0p1-tnone.parquet
seconds: 84.93752815400012
rows: 16580
queries: 3316
sha256: deca2f08d497fc46945cf4e852

/tmp/ipykernel_3583/2265487053.py:713: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  val_boundary = pd.concat(



Validation trajectory rows: (729520, 12)
Expected trajectory rows: 729520
Validation slope rows: (145904, 9)
Validation unique queries: 3316
Validation configs: 44
VALIDATION SLOPE RECONSTRUCTION — PASS

UNTOUCHED FEVER VALIDATION RESULTS
N: 145904
mean H3: 0.006771005926885983
median H3: 0.0
q75: 0.0
q90: 0.0
q95: 0.07381404638290404
q97.5: 0.1269576609134674

exact zero: 0.8797496984318456
near zero <=1e-12: 0.8975422195416164
positive H3: 0.0820950762144972
negative H3: 0.0381552253536572

stable/null: 0.8983920934312973
amplifying: 0.0794769163285448
reversal: 0.02213099024015791

VALIDATION POLICY-FAMILY SUMMARY


,method,observations,unique_queries,mean_H3,median_H3,exact_zero_fraction,positive_H3_fraction,amplifying_fraction,stable_or_null_fraction,reversal_fraction
0,mean,39792,3316,0.006237,0.0,0.877865,0.081951,0.079639,0.895029,0.025332
1,softmax,106112,3316,0.006971,0.0,0.880456,0.082149,0.079416,0.899653,0.020931



MOST AMPLIFYING VALIDATION CONFIGS


...


MOST STABLE VALIDATION CONFIGS


,method,alpha,k,temperature,config_key,observations,mean_H3,median_H3,exact_zero_fraction,positive_H3_fraction,amplifying_fraction,stable_or_null_fraction,reversal_fraction
13,softmax,0.1,5,0.10,softmax-k5-a0p1-t0p1,3316,0.001164,0.0,0.927021,0.033776,0.027141,0.967431,0.005428
0,mean,0.1,5,NaN,mean-k5-a0p1-tnone,3316,0.001105,0.0,0.926719,0.034379,0.027443,0.967129,0.005428
14,softmax,0.1,5,0.20,softmax-k5-a0p1-t0p2,3316,0.001129,0.0,0.927021,0.033474,0.027141,0.967129,0.005730
15,softmax,0.1,5,0.50,softmax-k5-a0p1-t0p5,3316,0.001149,0.0,0.926719,0.034077,0.027443,0.967129,0.005428
12,softmax,0.1,5,0.05,softmax-k5-a0p1-t0p05,3316,0.001222,0.0,0.926719,0.033474,0.027744,0.966224,0.006031
16,softmax,0.1,20,0.05,softmax-k20-a0p1-t0p05,3316,0.001461,0.0,0.924608,0.036490,0.031966,0.961701,0.006333
17,softmax,0.1,20,0.10,softmax-k20-a0p1-t0p1,3316,0.001541,0.0,0.924005,0.037093,0.033172,0.959590,0.007238
1,mean,0.1,20,NaN,mean-k20-a0p1-tnone,3316,0.001464,0.0,0.924005,0.037696,0.033776,0.959590,0.006634
19,softmax,0.1,20,0.50,softmax-k20-a0p1-t0p5,3316,0.001547,0.0,0.923703,0.037394,0.033776,0.959288,0.006936
18,softmax,0.1,20,0.20,softmax-k20-a0p1-t0p2,3316,0.001549,0.0,0.923703,0.037696,0.034077,0.958987,0.006936



VALIDATION QUERY SUSCEPTIBILITY
never amplifying: 0.8232810615199035
>=1 amplifying config: 0.1767189384800965
>=25% configs amplifying: 0.1284680337756333
>=50% configs amplifying: 0.07961399276236429



,query_id,configs,mean_H3,max_H3,amplification_rate,reversal_rate,exact_zero_rate
652,13942,44,0.155119,0.250000,1.000000,0.000000,0.000000
2274,33955,44,0.155119,0.250000,1.000000,0.000000,0.000000
917,157652,44,0.068479,0.169254,1.000000,0.000000,0.000000
2656,58707,44,0.078194,0.169254,1.000000,0.000000,0.000000
3099,87793,44,0.082548,0.169254,1.000000,0.000000,0.000000
1486,192725,44,0.070189,0.164871,1.000000,0.000000,0.000000
228,11310,44,0.107068,0.161315,1.000000,0.000000,0.000000
838,151529,44,0.086004,0.135621,1.000000,0.000000,0.000000
5,100234,44,0.035060,0.131546,1.000000,0.000000,0.000000
1700,202470,44,0.057440,0.124821,1.000000,0.000000,0.000000



ARC-v0.13 VALIDATION-ONLY CONTINUATION — PASS
Validation checkpoints: 44
Validation queries: 3316
Validation configs: 44
Exact zero: 0.8797496984318456
Stable/null: 0.8983920934312973
Amplifying: 0.0794769163285448
Reversal: 0.02213099024015791
Total continuation seconds: 3605.650623768
Validation report: /content/drive/MyDrive/rag-pq-checkpoints/arc-v0/fever-boundary-external-replication-v013/20260817-140640/v013_validation_continuation_report.json
Report SHA-256: 5b7e8bba20ae76c0171f1a6df90432b0f3cf03c1c2ef2a798e5153eab99d96be
